# Xenium Kidney – load from zarr

Read and process Xenium raw data files `cells.zarr` and `cell_feature_matrix.zarr`
instead of `cells.parquet` / `cell_feature_matrix.h5`.

Outputs `rna.h5ad` and `protein.h5ad` containing the **25 features** shared between
the RNA gene panel and the protein antibody panel (after mapping antibody names to
gene symbols), with Leiden cluster labels derived from the full 405-gene RNA matrix.

In [ ]:
import zarr
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad
import scanpy as sc

DATA_DIR = "/Users/sebgoti/Documents/PhD/TOAST_data/Xenium_V1_Human_Kidney_FFPE_Protein_updated_xe_outs"
OUT_DIR  = "/Users/sebgoti/Documents/PhD/TOAST_data"

# cells.zarr stores per-cell spatial and morphological metadata.
# The "cell_summary" array has shape (n_cells, 8): one row per cell,
# columns defined in cells.zarr/.zattrs → column_names.
cz = zarr.open(f"{DATA_DIR}/cells.zarr", mode="r")
cell_summary = np.array(cz["cell_summary"])   # float64, shape (465534, 8)

# Column order matches the "column_names" key in cells.zarr/.zattrs
col_names = [
    "cell_centroid_x", "cell_centroid_y",   # centroid in µm (used as spatial coords)
    "cell_area",                              # cell area in µm²
    "nucleus_centroid_x", "nucleus_centroid_y",
    "nucleus_area",
    "z_level",                               # z-stack level of the best focal plane
    "nucleus_count",                          # number of nuclei detected inside the cell
]
cells = pd.DataFrame(cell_summary, columns=col_names)
print(f"Loaded {len(cells):,} cells")
cells.head()

In [ ]:
# cell_feature_matrix.zarr stores the count matrix and feature metadata.
# Feature metadata lives in cell_features/.zattrs as JSON arrays (length 544):
#   feature_keys  – human-readable names (gene symbols, antibody names, control IDs)
#   feature_types – category string for each feature (see next cell for unique values)
#   feature_ids   – Ensembl IDs (genes) or internal target IDs (protein)
mz = zarr.open(f"{DATA_DIR}/cell_feature_matrix.zarr", mode="r")
attrs = dict(mz["cell_features"].attrs)

feature_keys  = np.array(attrs["feature_keys"])    # len 544
feature_types = np.array(attrs["feature_types"])   # len 544
feature_ids   = np.array(attrs["feature_ids"])     # len 544

# The count matrix is at cell_features/csc/.
# Despite the "csc" label, indptr has length n_cells + 1 (= 465535),
# meaning the pointer dimension runs over cells, not features.
# This is equivalent to a CSR matrix with rows = cells, columns = features,
# so we can pass the arrays directly to sp.csr_matrix without any transposition.
csc_grp = mz["cell_features/csc"]
mat = sp.csr_matrix(
    (np.array(csc_grp["data"]),            # non-zero count values
     np.array(csc_grp["indices"]).astype(np.int32),  # column (feature) indices
     np.array(csc_grp["indptr"])),         # row (cell) pointers
    shape=(len(np.array(csc_grp["indptr"])) - 1, len(feature_types)),
    # shape = (465534 cells, 544 features)
)
print(f"Matrix shape: {mat.shape}")

In [ ]:
# The 544 features break down into 6 types:
#   gene                    – 405  actual RNA transcripts targeted by the panel
#   negative_control_probe  –  20  probes that should not hybridise (RNA noise floor)
#   negative_control_codeword – 41 unused barcode slots (decoding noise floor)
#   unassigned_codeword     –  35  blank protein barcodes (protein noise floor)
#   deprecated_codeword     –  15  retired barcodes from previous panel versions
#   protein                 –  27  antibody-based protein markers
#   aggregate_gene          –   1  "Total transcripts" (sum across all RNA genes)
# We will only use "gene" (RNA) and "protein" columns for the alignment.
print(np.unique(feature_types))

In [ ]:
# Build a single AnnData that holds all 544 features.
# This mirrors what sc.read_10x_h5 returns in read.ipynb.
#
# obs  – one row per cell; obs_names are integer indices 0..n_cells-1 as strings
#         (Xenium zarr has no barcode strings like 'aaaaaglh-1', so we use integers)
# var  – one row per feature; var_names are the human-readable names from .zattrs
# obsm – "spatial" stores the (x, y) centroid coordinates in µm, used for plotting
adata = ad.AnnData(X=mat)
adata.obs_names  = pd.Index(np.arange(mat.shape[0]).astype(str))
adata.var_names  = pd.Index(feature_keys)          # gene symbols / antibody names / control IDs
adata.var["feature_types"] = feature_types         # used below to select genes vs proteins
adata.var["gene_ids"]      = feature_ids           # Ensembl IDs (genes) or TXP target IDs (protein)
adata.obsm["spatial"]      = cell_summary[:, :2]   # columns 0,1 = cell_centroid_x, cell_centroid_y

# Copy all cell metadata columns into obs so they travel with the AnnData
for col in col_names:
    adata.obs[col] = cells[col].values

adata

In [ ]:
# Sanity check: scatter plot of all 465k cells coloured uniformly.
# A kidney tissue section should show a clear boundary and internal structure.
# color=None means no gene/cluster colouring — just cell positions.
sc.pl.embedding(
    adata,
    basis="spatial",
    color=None,
    size=0.8,
    title="Xenium Spatial Plot",
)

In [ ]:
# Extract the 27 protein antibody columns and rename them to gene symbols.
#
# The Xenium protein panel uses commercial antibody names (e.g. "GranzymeB", "alphaSMA",
# "PD-1") which differ from the official HGNC gene symbols (GZMB, ACTA2, PDCD1).
# Renaming lets us compare protein and RNA features by the same name.
# Antibodies without an entry in rename_map already use the gene symbol (e.g. CD4, CD8A).
prot_mask     = adata.var["feature_types"] == "protein"
adata_protein = adata[:, prot_mask].copy()   # shape: (465534, 27)

rename_map = {
    "CD20":         "MS4A1",    # B-cell marker
    "CD11c":        "ITGAX",    # dendritic cell / macrophage marker
    "CD138":        "SDC1",     # plasma cell marker
    "GranzymeB":    "GZMB",     # cytotoxic T-cell / NK cell effector
    "CD16":         "FCGR3A",   # NK cell / neutrophil Fc receptor
    "Ki-67":        "MKI67",    # proliferation marker
    "PD-1":         "PDCD1",    # T-cell exhaustion checkpoint
    "PD-L1":        "CD274",    # checkpoint ligand (tumour / immune cells)
    "VISTA":        "VSIR",     # immune checkpoint receptor
    "LAG-3":        "LAG3",     # T-cell exhaustion checkpoint
    "CD31":         "PECAM1",   # endothelial / platelet marker
    "Beta-catenin": "CTNNB1",   # Wnt signalling / epithelial junction
    "E-Cadherin":   "CDH1",     # epithelial cell adhesion
    "Vimentin":     "VIM",      # mesenchymal / stromal marker
    "alphaSMA":     "ACTA2",    # smooth muscle / myofibroblast marker
    "CD45":         "PTPRC",    # pan-leukocyte marker
}

adata_protein.var_names = pd.Index(
    [rename_map.get(n, n) for n in adata_protein.var_names]
)
adata_protein.var_names

In [ ]:
# Find genes that appear in BOTH the RNA panel and the protein antibody panel.
# After renaming, 25 of the 27 protein targets share a name with an RNA gene
# (2 protein targets — HLA-DR and PanCK — have no direct RNA gene symbol equivalent).
# These 25 shared features are the cross-modal anchor: for each cell we have
# both a mRNA count and an antibody count for the same gene product,
# which makes the expression cost matrix M biologically meaningful in the alignment.
rna_mask = adata.var["feature_types"] == "gene"
genes    = set(adata.var_names[rna_mask])     # 405 RNA genes
proteins = set(adata_protein.var_names)       # 27 antibody targets (after renaming)
intersection = genes.intersection(proteins)   # expected: 25 shared gene symbols
print("Number of overlapping names:", len(intersection))
print(sorted(intersection))

In [ ]:
# Create two AnnData objects restricted to the 25 shared features.
# Both have identical obs_names (same cells, same order) — only the measurement
# modality differs: adata_rna_intersect.X holds mRNA counts,
# adata_protein_intersect.X holds antibody counts for the same 25 gene products.
# This shared feature space allows the alignment to compute a meaningful
# expression cost matrix M = cdist(rna_X, protein_X) across modalities.
#
# Note: adata_rna (full 405 genes) is kept separately for clustering below.
shared = list(intersection)   # 25 gene symbols; order will be the same in both subsets

adata_rna               = adata[:, rna_mask].copy()         # full RNA: (465534, 405)
adata_rna_intersect     = adata_rna[:, shared].copy()       # RNA subset: (465534, 25)
adata_protein_intersect = adata_protein[:, shared].copy()   # protein subset: (465534, 25)

print(adata_rna_intersect)
print(adata_protein_intersect)

In [ ]:
# Cluster cells using the FULL 405-gene RNA matrix (not the 25-gene subset).
# More features give a richer representation of cell identity.
# The resulting "unsupervised_label" will be copied to both intersect AnnDatas
# and is used by the TOAST alignment as the cell-type label for spatial entropy
# (cost term C3) and average-neighbour expression (cost term C4).
#
# Pipeline:
#   scale    – z-score each gene across cells, clip at ±10 to reduce outlier influence
#   pca      – reduce to 50 PCs (default); captures most variance across 405 genes
#   neighbors – build a k-NN graph in PCA space (k=15 by default)
#   leiden   – graph community detection; resolution=0.2 gives ~18 coarse clusters
sc.pp.scale(adata_rna, max_value=10)
sc.tl.pca(adata_rna)
sc.pp.neighbors(adata_rna)
sc.tl.leiden(adata_rna, resolution=0.2, key_added="unsupervised_label")

In [ ]:
# Verify that all three AnnDatas cover exactly the same cells in the same order,
# then copy the Leiden labels into both intersect AnnDatas.
# The assertion is a safety check: if cell ordering ever differed between the
# full-RNA AnnData and the intersect AnnDatas, the copied labels would be wrong.
assert list(adata_rna.obs_names) == list(adata_protein_intersect.obs_names)
assert list(adata_rna.obs_names) == list(adata_rna_intersect.obs_names)
print("Cell order matches")

# Both intersect AnnDatas now carry the same "unsupervised_label" column.
# The alignment notebook reads this column to compute TOAST's spatial-entropy
# and neighbour-expression cost terms.
adata_protein_intersect.obs["unsupervised_label"] = adata_rna.obs["unsupervised_label"].copy()
adata_rna_intersect.obs["unsupervised_label"]     = adata_rna.obs["unsupervised_label"].copy()

In [ ]:
# Write both intersect AnnDatas to disk.
# The MultimodalAlignment notebook loads these files as its starting point.
# Each file contains:
#   X            – raw counts for the 25 shared features (sparse CSR)
#   obs          – cell metadata + "unsupervised_label" (Leiden cluster id)
#   var          – feature names and metadata for the 25 shared genes
#   obsm/spatial – (x, y) centroid coordinates in µm
adata_rna_intersect.write(f"{OUT_DIR}/rna.h5ad")
adata_protein_intersect.write(f"{OUT_DIR}/protein.h5ad")
print(f"Saved rna.h5ad and protein.h5ad to {OUT_DIR}")

In [ ]:
# Print the cluster IDs to confirm how many Leiden communities were found.
# At resolution=0.2 this should be ~18 clusters, matching the original read.ipynb output.
# Fewer clusters → coarser cell-type resolution in the alignment cost terms.
# If you need finer resolution (more clusters), increase the leiden resolution parameter above.
print("Clusters found:", adata_rna_intersect.obs["unsupervised_label"].unique())